In [1]:
import pandas as pd

df = pd.read_csv("trial_data.csv")
df.shape

(2000, 10)

In [2]:
!pip install lifelines

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-win_amd64.whl.metadata (52 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached wrapt-2.1.1-cp311-cp311-win_amd64.whl.metadata (7.6 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp311-cp311-win_amd64.whl.metadata (116 kB)
  Using cached kiwisolver-1.4.9-cp311-cp311-win_amd64.whl.metadata (6.4 kB)
Using cached panda

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.


In [3]:
from lifelines import CoxPHFitter

cph = CoxPHFitter()
cph.fit(df, duration_col="time", event_col="event")

cph.print_summary()

<lifelines.CoxPHFitter: fitted with 2000 total observations, 1373 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 2000
number of events observed = 627
   partial log-likelihood = -4603.19
         time fit was run = 2026-02-27 08:27:47 UTC

---
             coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                    
treatment   -0.29      0.75      0.08           -0.45           -0.13                0.64                0.87
age          0.00      1.00      0.00           -0.00            0.01                1.00                1.01
hr_status    0.30      1.35      0.23           -0.15            0.75                0.86                2.13
her2_status  0.23      1.25      0.11            0.00            0.45                1.00                1.57
tnbc         0.94      2.57      0.25            0.45            1.44                1.57                4.20
tumor_size   0.23      1.25      0.06            0.12            0.34                1.12                1.40
nodes        0.28      1.32      0.05            0.17            0.38                1.19                1.46
grade        0.13      1.14      0.06            0.01            0.25                1.01                1.28

             cmp to     z      p  -log2(p)
covariate                                 
treatment      0.00 -3.63 <0.005     11.79
age            0.00  1.12   0.26      1.94
hr_status      0.00  1.30   0.20      2.36
her2_status    0.00  1.96   0.05      4.32
tnbc           0.00  3.77 <0.005     12.60
tumor_size     0.00  3.99 <0.005     13.86
nodes          0.00  5.21 <0.005     22.36
grade          0.00  2.13   0.03      4.92
---
Concordance = 0.61
Partial AIC = 9222.37
log-likelihood ratio test = 104.44 on 8 df
-log2(p) of ll-ratio test = 60.72

In [4]:
cph.concordance_index_

np.float64(0.6134466977321444)

In [5]:
!pip install scikit-survival

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/802.0 kB ? eta -:--:--
   ---------------------------------------- 802.0/802.0 kB 6.9 MB/s  0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.1 MB 3.8 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.1 MB 3.8 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.1 MB 3.8 MB/s eta 0:00:02
   ------------------- -------------------- 3.9/8.1 MB 3.9 MB/s eta 0:00:02
   ---------------------- ----------------- 4.5/8.1 MB 3.8 MB/s eta 0:00:01
   --------------------------- ------------ 5.5/8.1 MB 3.9 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.1 MB 3.9 MB/s eta 0:00:01
   ------------------------------------ --- 7.3/8.1 MB 3.9 MB/s eta 0:00:01
   -------------------------

In [6]:
import numpy as np
from sksurv.util import Surv

# Define outcome
y = Surv.from_dataframe("event", "time", df)

# Define features (drop time & event)
X = df.drop(columns=["time", "event"])

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [8]:
from sksurv.ensemble import RandomSurvivalForest

rsf = RandomSurvivalForest(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

,n_estimators,200
,max_depth,None
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,42


In [9]:
from sksurv.metrics import concordance_index_censored

# Predict risk scores
risk_scores = rsf.predict(X_test)

c_index = concordance_index_censored(
    y_test["event"],
    y_test["time"],
    risk_scores
)

c_index

(np.float64(0.5846149633037573),
 np.int64(53366),
 np.int64(37917),
 np.int64(7),
 np.int64(0))